In [1]:
from pathlib import Path

DATA_DIR = Path("output")

OUTPUT_DIR = Path("model_outputs")

RANDOM_STATE = 42

In [2]:
dataset_path = Path("output/dataset_winsize5h_where.csv")

In [3]:
import json

import pandas as pd

where_df = pd.read_csv(
    dataset_path,
    dtype={"fold_id": "int64"},
    converters={
        "window": json.loads,
        "label": json.loads,
    },
)

where_df.head()

,fold_id,window,label
0,0,"[{'from_zone_index': 16, 'to_zone_index': 16, ...","{'from_zone_index': 12, 'to_zone_index': 12}"
1,3,"[{'from_zone_index': 9, 'to_zone_index': 9, 'o...","{'from_zone_index': 17, 'to_zone_index': 17}"
2,2,"[{'from_zone_index': 20, 'to_zone_index': 20, ...","{'from_zone_index': 2, 'to_zone_index': 2}"
3,2,"[{'from_zone_index': 9, 'to_zone_index': 20, '...","{'from_zone_index': 22, 'to_zone_index': 22}"
4,1,"[{'from_zone_index': 2, 'to_zone_index': 2, 'o...","{'from_zone_index': 9, 'to_zone_index': 9}"


In [4]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

OUTAGE_TYPE_TO_ID = {
    "Planned": 0,
    "Auto": 1,
}


class WhereOutageDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        window = row["window"]
        label = row["label"]

        outage_type = torch.tensor(
            [OUTAGE_TYPE_TO_ID[event["outage_type"]] for event in window],
            dtype=torch.long,
        )
        from_zone_indices = torch.tensor(
            [event["from_zone_index"] for event in window],
            dtype=torch.long,
        )
        to_zone_indices = torch.tensor(
            [event["to_zone_index"] for event in window],
            dtype=torch.long,
        )
        target = torch.tensor(
            [label["from_zone_index"], label["to_zone_index"]],
            dtype=torch.long,
        )

        return {
            "outage_type": outage_type,
            "from_zone_indices": from_zone_indices,
            "to_zone_indices": to_zone_indices,
            "target": target,
        }


TRAIN_FOLDS = [0, 1, 2, 3, 4]

train_where_dataset = WhereOutageDataset(
    where_df[where_df["fold_id"].isin(TRAIN_FOLDS)]
)

In [5]:
for i, sample_dict in enumerate(train_where_dataset):
    print(sample_dict)
    if i == 10:
        break

{'outage_type': tensor([0]), 'from_zone_indices': tensor([16]), 'to_zone_indices': tensor([16]), 'target': tensor([12, 12])}
{'outage_type': tensor([0]), 'from_zone_indices': tensor([9]), 'to_zone_indices': tensor([9]), 'target': tensor([17, 17])}
{'outage_type': tensor([0, 0]), 'from_zone_indices': tensor([20, 16]), 'to_zone_indices': tensor([20, 16]), 'target': tensor([2, 2])}
{'outage_type': tensor([0, 0, 0, 0, 0, 0, 0]), 'from_zone_indices': tensor([ 9,  2,  2, 11,  2,  2, 20]), 'to_zone_indices': tensor([20,  2,  2, 11,  2,  2, 20]), 'target': tensor([22, 22])}
{'outage_type': tensor([0, 0]), 'from_zone_indices': tensor([ 2, 20]), 'to_zone_indices': tensor([ 2, 20]), 'target': tensor([9, 9])}
{'outage_type': tensor([1, 0, 0]), 'from_zone_indices': tensor([9, 3, 0]), 'to_zone_indices': tensor([9, 3, 0]), 'target': tensor([8, 8])}
{'outage_type': tensor([0, 0]), 'from_zone_indices': tensor([2, 2]), 'to_zone_indices': tensor([2, 2]), 'target': tensor([15, 15])}
{'outage_type': tensor

In [ ]:
import torch.nn as nn

ZONE_EMBEDDING_DIM = 64
OUTAGE_TYPE_EMBEDDING_DIM = 16
TRANSFORMER_DIM = 128
DROPOUT = 0.1

num_zones = int(
    max(
        where_df["label"]
        .map(lambda x: max(x["from_zone_index"], x["to_zone_index"]))
        .max(),
        max(
            max(event["from_zone_index"], event["to_zone_index"])
            for window in where_df["window"]
            for event in window
        ),
    )
    + 1
)
num_outage_types = len(OUTAGE_TYPE_TO_ID)


class WhereTransformer(nn.Module):
    def __init__(
        self,
        num_zones,
        num_outage_types,
        zone_embedding_dim=ZONE_EMBEDDING_DIM,
        outage_type_embedding_dim=OUTAGE_TYPE_EMBEDDING_DIM,
        transformer_dim=TRANSFORMER_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()
        self.num_zones = num_zones
        self.zone_embedding = nn.Embedding(num_zones, zone_embedding_dim)
        self.outage_type_embedding = nn.Embedding(
            num_outage_types,
            outage_type_embedding_dim,
        )
        self.input_fc = nn.Linear(
            2 * zone_embedding_dim + outage_type_embedding_dim,
            transformer_dim,
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=transformer_dim,
            nhead=transformer_dim // 64,
            dim_feedforward=transformer_dim * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=1,
        )
        self.projection = nn.Linear(transformer_dim, 2 * num_zones)

    def forward(self, from_zone_indices, to_zone_indices, outage_type):
        from_zone_emb = self.zone_embedding(from_zone_indices)
        to_zone_emb = self.zone_embedding(to_zone_indices)
        outage_type_emb = self.outage_type_embedding(outage_type)

        x = torch.cat([from_zone_emb, to_zone_emb, outage_type_emb], dim=-1)
        x = self.input_fc(x)
        x = self.transformer_encoder(x)
        pooled = x.mean(dim=1)
        logits = self.projection(pooled)
        return logits.view(-1, 2, self.num_zones)


where_transformer = WhereTransformer(
    num_zones=num_zones,
    num_outage_types=num_outage_types,
)
where_transformer

WhereTransformer(
  (zone_embedding): Embedding(24, 64)
  (outage_type_embedding): Embedding(2, 16)
  (input_fc): Linear(in_features=144, out_features=128, bias=True)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (projection): Linear(in_features=128, out_features=48, bias=True)
)